In [1]:
from typing import List, Optional, Literal, Dict, Any, Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

# Demo 1: State only (no memory

In [6]:
class SimpleState(TypedDict):
    messages: List[str]
    user_name: str
    counter: int

def process_without_memory(state: SimpleState) -> SimpleState:
    state['counter'] += 1
    state['messages'].append(f"Processed at count {state['counter']}")
    return state

workflow = StateGraph(SimpleState)
workflow.add_node("process", process_without_memory)
workflow.add_edge(START, "process")
workflow.add_edge("process", END)
app = workflow.compile()

{'messages': ['Processed at count 1'], 'user_name': 'Alice', 'counter': 1}

In [14]:
app.invoke({"messages": [], "user_name": "Alice", "counter": 0})
app.invoke({"messages": [], "user_name": "Alice", "counter": 0})

{'messages': ['Processed at count 1'], 'user_name': 'Alice', 'counter': 1}

# Demo 2: With checkpointer (memory)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
workflow = StateGraph(SimpleState)
workflow.add_node("process", process_without_memory)
workflow.add_edge(START, "process")
workflow.add_edge("process", END)
app = workflow.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "conversation_123"}}
result1 = app.invoke({"messages": [], "user_name": "Alice", "counter": 0}, config=config)

saved_state = app.get_state(config).values
result2 = app.invoke(saved_state, config=config)